# Notebook 03 — SageMaker Pipeline with Post-Pipeline SageMaker MLflow App Logging

**Module:** ITI113 Machine Learning & Operations  
**Focus Area:** C — MLOps (Pipeline, experiment tracking, CI/CD, deployment)  
**Estimated Runtime:** 15–25 minutes for the pipeline execution

---

## What this notebook does

1. Writes `preprocess.py`, `train.py`, and `inference.py` to a local `src/` folder.
2. Defines and runs a SageMaker Pipeline: **Process → Train → Condition → Register**.
3. Keeps the SageMaker training container free of MLflow credentials and MLflow dependencies.
4. After a successful pipeline run, the notebook reads SageMaker job metadata and metrics, then logs them to the team’s **SageMaker Serverless MLflow App**. The notebook validates the app `TeamId` tag before logging.
5. Registers a quality-approved model in **SageMaker Model Registry**.

This separation makes troubleshooting safer and simpler:

```text
SageMaker Pipeline                    Notebook-side MLflow logging
Process → Train → Gate → Register     Read run metadata → Log to MLflow App
```

This version follows Notebook 01/02 and uses the SageMaker MLflow App ARN when available from `mlflow_app_config_team01_s004.json`.


In [1]:
# After running, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow


Note: you may need to restart the kernel to use updated packages.


## 0. Configuration

In [10]:
import boto3
import sagemaker
import json
import os
import time
from pathlib import Path

# ----------------------------
# AWS / SageMaker setup
# ----------------------------
session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

# Use the ITI113 course bucket and team prefix.
# This matches the team execution role S3 policy, e.g.:
# s3://nyp-26s1-iti113/iti113/team40/
BUCKET  = "nyp-26s1-iti113"

# Change these for the current student/profile.
TEAM_ID = "team07"
STUDENT_ID = "s703"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "heart-disease"

PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

# Instance types.
# If ml.m5.large quota is 0, change these to an approved available training/processing type.
# For ITI113, keep Studio spaces on ml.t3.medium and use SageMaker jobs for training/processing.
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

# ----------------------------
# SageMaker Serverless MLflow App setup
# ----------------------------
# Preferred:
# 1. Team-level config copied/shared from Notebook 01A:
#       mlflow_app_config_team40.json
# 2. Student-specific config from Notebook 01A:
#       mlflow_app_config_team40_s4002.json
# 3. Any local config matching this team:
#       mlflow_app_config_team40_*.json
#
# Important:
# Do not use another team's MLflow ARN. With team-level IAM
# restriction, wrong-team access should fail with 403.
# ----------------------------

TEAM_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

config_candidates = [
    TEAM_CONFIG_FILE,
    STUDENT_CONFIG_FILE,
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]

MLFLOW_APP_ARN = None
MLFLOW_EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment1"
mlflow_config = {}
config_used = None

for config_file in config_candidates:
    if config_file.exists():
        mlflow_config = json.loads(config_file.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = (
            mlflow_config.get("MLFLOW_APP_ARN")
            or mlflow_config.get("mlflow_app_arn")
            or mlflow_config.get("arn")
        )
        MLFLOW_EXPERIMENT_NAME = (
            mlflow_config.get("EXPERIMENT_NAME")
            or mlflow_config.get("experiment_name")
            or MLFLOW_EXPERIMENT_NAME
        )
        config_used = config_file
        break

# Fallback for classroom testing only.
# Update this to your team's MLflow App ARN from Notebook 01A if the config file
# is not available in this Studio workspace.
DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-L5IMA5YSBDTY"
)

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print(
        "[WARNING] No local MLflow config file found. "
        "Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team."
    )
else:
    print(f"Loaded MLflow App config from {config_used}")

# Validate config team, if present.
config_team_id = mlflow_config.get("TEAM_ID") or mlflow_config.get("team_id")
if config_team_id and config_team_id != TEAM_ID:
    raise ValueError(
        f"Config file team mismatch: config TEAM_ID={config_team_id}, notebook TEAM_ID={TEAM_ID}. "
        "Do not use another team's MLflow config."
    )

# ----------------------------
# Safety check for team-level MLflow restriction
# ----------------------------
# The selected MLflow App must have ResourceTag/TeamId = TEAM_ID.
# If a student accidentally uses another team's ARN, this should either:
# - fail with AccessDenied / 403 due to IAM restriction, or
# - fail this explicit validation before logging.
# ----------------------------

sm_for_mlflow = boto3.client("sagemaker", region_name=region)

try:
    tag_response = sm_for_mlflow.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}

    print("MLflow App tags:")
    for k, v in mlflow_app_tags.items():
        print(f"  {k}: {v}")

    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App."
        )

    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")

except Exception as e:
    print("\n[ERROR] Could not validate MLflow App team tag.")
    print("This usually means one of the following:")
    print("1. The MLflow App ARN belongs to another team and IAM correctly blocked access.")
    print("2. The MLflow App is missing the TeamId tag.")
    print("3. The current role lacks permission to list tags for this MLflow App.")
    print(type(e).__name__, e)
    raise

# Optional: store for downstream cells and subprocesses.
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_APP_ARN
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME


# ----------------------------
# MLflow App UI link helpers
# ----------------------------
# MLflow may print generic links such as:
# https://mlflow.sagemaker.ap-southeast-1.app.aws/#/...
# Those links are not presigned and may show a permission/session error.
# Use these helpers to generate a fresh presigned SageMaker MLflow App URL
# and append the experiment/run fragment.

def create_mlflow_app_presigned_url(fragment: str = "") -> str:
    sm_for_mlflow = boto3.client("sagemaker", region_name=region)
    response = sm_for_mlflow.create_presigned_mlflow_app_url(
        Arn=MLFLOW_APP_ARN
    )

    base_url = response.get("AuthorizedUrl") or response.get("Url")

    if not base_url:
        raise RuntimeError(
            "create_presigned_mlflow_app_url did not return AuthorizedUrl or Url. "
            f"Response: {response}"
        )

    # Remove any existing fragment before appending our own MLflow UI route.
    base_url = base_url.split("#", 1)[0]

    if fragment:
        return base_url + "#" + fragment.lstrip("#")

    return base_url


def print_mlflow_presigned_links(experiment_id=None, run_id=None):
    if experiment_id is not None:
        experiment_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}"
        )
        print("Presigned MLflow experiment URL:")
        print(experiment_url)

    if experiment_id is not None and run_id is not None:
        run_url = create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}/runs/{run_id}"
        )
        print("\nPresigned MLflow run URL:")
        print(run_url)

PIPELINE_NAME       = f"iti113-{TEAM_ID}-heart-disease"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-HeartDisease"
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-heart-disease"
QUALITY_GATE_AUC    = 0.75

RAW_DATA_URI  = f"s3://{BUCKET}/{PREFIX}/raw/heart.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"

# Store the pipeline source files in S3 first, then download them into a clean local folder.
SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI    = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

print(f"Pipeline                : {PIPELINE_NAME}")
print(f"Bucket                  : {BUCKET}")
print(f"Team prefix             : {PREFIX}")
print(f"Semester                : {SEMESTER}")
print(f"Region                  : {region}")
print(f"SageMaker role          : {role}")
print(f"MLflow App ARN          : {MLFLOW_APP_ARN}")
print(f"MLflow experiment       : {MLFLOW_EXPERIMENT_NAME}")
print(f"Pipeline source S3 URI  : {SCRIPTS_S3_URI}")
print(f"Local pipeline source   : {LOCAL_PIPELINE_SRC}")


Loaded MLflow App config from mlflow_app_config_team07_s703.json
MLflow App tags:
  Semester: 26S1
  sagemaker:domain-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-cydcdxkc4yot
  ProjectName: heart-disease
  sagemaker:space-arn: arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-cydcdxkc4yot/team07-shared
  Course: ITI113
  TeamId: team07
  CreatedByNotebook: 01A_setup_sagemaker_mlflow_app
  StudentId: s703
[OK] MLflow App tag TeamId=team07 matches notebook TEAM_ID=team07
Pipeline                : iti113-team07-heart-disease
Bucket                  : nyp-26s1-iti113
Team prefix             : iti113/team07/data/heart-disease
Semester                : 26S1
Region                  : ap-southeast-1
SageMaker role          : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team07
MLflow App ARN          : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O
MLflow experiment       : ITI113/team07/Experiment1
Pipeline source S3 URI  : s3://ny

## 0A. Precheck SageMaker MLflow App connection

Before launching the SageMaker Pipeline, test that this notebook can create/use the **current team's** MLflow experiment in the SageMaker MLflow App.

Example for Team 40:

```text
ITI113/team40/Experiment1
```

This notebook now checks that the selected MLflow App has the correct `TeamId` tag before logging. If this fails, resolve the MLflow App ARN, `sagemaker-mlflow` package, or IAM permissions before continuing to the SageMaker Pipeline.


### Note about MLflow links

The MLflow client may print links such as `https://mlflow.sagemaker.ap-southeast-1.app.aws/#/...`. Those generic links are not presigned and may show a SageMaker MLflow permission/session error. This notebook generates fresh presigned MLflow App URLs using `create_presigned_mlflow_app_url()` after each logging step. Use those printed presigned URLs instead.


In [11]:
import mlflow
import time

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{TEAM_ID}_pipeline_notebook_precheck_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "heart-disease",
        "run_type": "sagemaker_pipeline_precheck",
        "tracking_backend": "sagemaker_mlflow_app",
        "mlflow_app_arn": MLFLOW_APP_ARN,
    })
    mlflow.log_param("source", "notebook_03_precheck")
    mlflow.log_metric("connection_success", 1)

    precheck_run_id = run.info.run_id
    precheck_experiment_id = run.info.experiment_id

print("SageMaker MLflow App precheck completed.")
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", precheck_experiment_id)
print("Run ID:", precheck_run_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=precheck_experiment_id,
    run_id=precheck_run_id
)


🏃 View run team07_pipeline_notebook_precheck_1784876937 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/8177cf21390a43efaf28a03120bc73f3
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App precheck completed.
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O
Experiment: ITI113/team07/Experiment1
Experiment ID: 1
Run ID: 8177cf21390a43efaf28a03120bc73f3

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-PVSI6X27672O.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkFDNjRNNyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNGpkNURDVU8wOW9INkNnVGFiSjBrRzJMT3JvKzBLM0p2NThFQ1huc0xJUjhBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGcU1rSlVVRFJVYldWVWN6VnZaR1JRZEhrM1lYTjBOR1ZFUmtoaE4ybDBNaXR6TjJJeldFbHVaVUY1ZVVKc2RYUjVhV2syS3pSTFdXVmlOVWMxU0Zab2R6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFjdzhLbUFBeXdVZ2VKVy8zenZhbjJ3QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF5MTJtenJ2ZjZTTUpVcm1iTUNBUkNBTzUzNG1BMmEwekNraDBHNjJNb0E4RjNMOCsvZ1lHY0FlQnhDUjdtM3pRRkszVkdTbDFLaWdrWkRRUmhGd1JlMUVzdHFkQVR5UmNxbWthbWpBZ0FBRUFDUm42eGljQzVCNk1hQWdNT0VCSkkzK0RWUGJRc2c2R1E5RDdVLzRUMjlLT0lDVkVNM284Q0


Presigned MLflow run URL:
https://app-PVSI6X27672O.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkhJR1lETSIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNEdWN3dManVRcU5VTG1Lem9TT01Nb1VSbk9DV1NGZ0c4WFFWOG5TMGtnazBBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGd2RETlJXSEpvYWt4NFVsZFRlamx4YWswcldWaFpjRWdyVEZCU1lsbEpaWFY0V2swdlpXZExjQ3NyTTJ3cmVuZG1hbGN2YUNzMldFbzFPVFZUWW5nd1VUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFhQnk1ZS9ZMlRpSTBXckhtdHZkTU84QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4Z0NLZitycDRYdzhoeFhFVUNBUkNBTzhPbHVDK1h0cFJKd01sbXNERWtSeXcyaitZeE1SMUhGV3ZpbExmbHQyWFFERHpWM1QrRXpQeEJ4Y3VEMHZPdUdNeE1iSTJOOUJCbWN1MWRBZ0FBRUFCMUh1Y0pNNEpNeE5HQm1vU3dKWlB4K0RFY21LeDNmWE9qNklwRGtjSm1rRStHVE8yQlNndUJJYnNE

## 1. Write Pipeline Scripts

SageMaker Pipeline steps run as isolated jobs, each in its own managed container.

The training script deliberately does **not** import or call MLflow. It only trains the model, prints quality metrics for the SageMaker quality gate, and saves `model.pkl` for SageMaker Model Registry.

After the pipeline completes, this notebook logs the completed run to the team SageMaker MLflow App experiment from the notebook environment. This means the MLflow App access stays in the notebook session and is not sent to the SageMaker training container.

After writing these files locally, the next sections upload them to S3 and download them into `pipeline_src/`. The pipeline then uses `pipeline_src/`, not the original `src/` folder.


In [12]:
os.makedirs('src', exist_ok=True)
print('src/ directory ready')


src/ directory ready


In [13]:
%%writefile src/preprocess.py
"""SageMaker Processing Job — replicates Notebook 01 preprocessing."""
import os, argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

parser = argparse.ArgumentParser()
parser.add_argument('--test-size',    type=float, default=0.20)
parser.add_argument('--random-state', type=int,   default=42)
args = parser.parse_args()

input_path = '/opt/ml/processing/input/heart.csv'
output_dir = '/opt/ml/processing/output'
os.makedirs(output_dir, exist_ok=True)

COLUMNS = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','target']
NUMERIC     = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL = ['sex','cp','fbs','restecg','exang','slope','ca','thal']

df = pd.read_csv(
    input_path,
    names=COLUMNS,
    header=None,
    na_values=["?", "", "NA", "N/A"]
)

# Convert every expected dataset column to numeric.
# This also turns an accidental header row or invalid text into NaN.
for col in COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Target must exist before binary conversion.
df = df.dropna(subset=["target"]).copy()
df["target"] = (df["target"] > 0).astype(int)

# Fill missing numeric values using the training-data median.
for col in NUMERIC:
    median_value = df[col].median()

    if pd.isna(median_value):
        raise ValueError(f"Column '{col}' has no valid numeric values.")

    df[col] = df[col].fillna(median_value)

# Fill missing categorical values using the most frequent category.
for col in CATEGORICAL:
    mode_values = df[col].mode(dropna=True)

    if mode_values.empty:
        raise ValueError(f"Column '{col}' has no valid categorical values.")

    df[col] = df[col].fillna(mode_values.iloc[0])

# Remove any remaining invalid records before feature engineering.
df = df.dropna(subset=NUMERIC + CATEGORICAL + ["target"]).copy()

print(f"Loaded {df.shape[0]} valid rows")
print("Column data types:")
print(df.dtypes)

# Use open-ended bins so an unexpected age does not produce NaN.
df["age_group"] = pd.cut(
    df["age"],
    bins=[float("-inf"), 45, 60, float("inf")],
    labels=[0, 1, 2],
    include_lowest=True
).astype(int)

df["high_risk_count"] = (
    (df["exang"] == 1).astype(int)
    + (df["fbs"] == 1).astype(int)
    + (df["ca"] > 0).astype(int)
)

FEATURES = NUMERIC + CATEGORICAL + ["age_group", "high_risk_count"]
X, y = df[FEATURES], df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=args.test_size, random_state=args.random_state, stratify=y)

scaler = StandardScaler()
X_train[NUMERIC] = scaler.fit_transform(X_train[NUMERIC])
X_test[NUMERIC]  = scaler.transform(X_test[NUMERIC])

X_train.to_csv(f'{output_dir}/train_features.csv', index=False)
y_train.to_csv(f'{output_dir}/train_labels.csv',   index=False, header=True)
X_test.to_csv( f'{output_dir}/test_features.csv',  index=False)
y_test.to_csv( f'{output_dir}/test_labels.csv',    index=False, header=True)
print('Preprocessing complete. Saved 4 output files.')


Overwriting src/preprocess.py


In [14]:
%%writefile src/train.py
"""
SageMaker Training Job
----------------------
Trains a Random Forest classifier and saves the SageMaker model artefact.

MLflow logging is intentionally performed outside the SageMaker training
container by the notebook after a successful pipeline execution.
"""
import os
import argparse
import pickle
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
)


parser = argparse.ArgumentParser()
parser.add_argument("--n-estimators", type=int, default=100)
parser.add_argument("--max-depth", type=int, default=6)
parser.add_argument("--min-samples-leaf", type=int, default=4)
parser.add_argument("--random-state", type=int, default=42)

# Metadata is retained in the SageMaker training job hyperparameters.
parser.add_argument("--team-id", type=str, default=os.environ.get("TEAM_ID", "unknown-team"))
parser.add_argument("--student-id", type=str, default=os.environ.get("STUDENT_ID", "s000"))
parser.add_argument("--semester", type=str, default=os.environ.get("SEMESTER", "26S1"))
parser.add_argument("--run-name", type=str, default="sagemaker_pipeline_run")

# SageMaker supplies these paths automatically.
parser.add_argument(
    "--model-dir",
    type=str,
    default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
)
parser.add_argument(
    "--train",
    type=str,
    default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train")
)
parser.add_argument(
    "--test",
    type=str,
    default=os.environ.get("SM_CHANNEL_TEST", "/opt/ml/input/data/test")
)
args = parser.parse_args()

os.makedirs(args.model_dir, exist_ok=True)

print("=== SageMaker Training Environment ===")
print(f"Train channel: {args.train}")
print(f"Test channel: {args.test}")
print(f"Model directory: {args.model_dir}")

X_train = pd.read_csv(os.path.join(args.train, "train_features.csv"))
y_train = pd.read_csv(os.path.join(args.train, "train_labels.csv")).squeeze("columns")
X_test = pd.read_csv(os.path.join(args.test, "test_features.csv"))
y_test = pd.read_csv(os.path.join(args.test, "test_labels.csv")).squeeze("columns")

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

if len(pd.Series(y_train).unique()) < 2:
    raise ValueError("Training labels contain fewer than two classes.")

model = RandomForestClassifier(
    n_estimators=args.n_estimators,
    max_depth=args.max_depth,
    min_samples_leaf=args.min_samples_leaf,
    class_weight="balanced",
    random_state=args.random_state,
    n_jobs=-1,
)
model.fit(X_train, y_train)

all_metrics = {}
for split, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    all_metrics.update({
        f"{split}_accuracy": round(accuracy_score(y, predictions), 4),
        f"{split}_f1": round(f1_score(y, predictions, zero_division=0), 4),
        f"{split}_precision": round(
            precision_score(y, predictions, zero_division=0), 4
        ),
        f"{split}_recall": round(
            recall_score(y, predictions, zero_division=0), 4
        ),
    })

    # AUC needs both classes in the evaluated split.
    if len(pd.Series(y).unique()) >= 2:
        all_metrics[f"{split}_auc_roc"] = round(
            roc_auc_score(y, probabilities), 4
        )
    else:
        all_metrics[f"{split}_auc_roc"] = None
        print(f"Warning: {split} split has only one class; AUC-ROC unavailable.")

print("=== Metrics ===")
for metric_name, metric_value in all_metrics.items():
    print(f"{metric_name}: {metric_value}")

with open(os.path.join(args.model_dir, "model.pkl"), "wb") as f:
    pickle.dump(model, f)

print(f"Model saved: {os.path.join(args.model_dir, 'model.pkl')}")

# These exact printed labels are captured by SageMaker metric_definitions
# and used by the Pipeline ConditionStep.
if all_metrics["test_auc_roc"] is None:
    raise ValueError("Test AUC-ROC is unavailable; cannot evaluate the quality gate.")

print(f"Test AUC-ROC: {all_metrics['test_auc_roc']}")
print(f"test_accuracy: {all_metrics['test_accuracy']}")
print(f"test_f1: {all_metrics['test_f1']}")


Overwriting src/train.py


In [15]:
%%writefile src/inference.py
"""SageMaker inference handler for the deployed endpoint."""
import os, json, pickle
import pandas as pd

FEATURE_COLUMNS = [
    'age','trestbps','chol','thalach','oldpeak',
    'sex','cp','fbs','restecg','exang','slope','ca','thal',
    'age_group','high_risk_count'
]

def model_fn(model_dir):
    with open(os.path.join(model_dir,'model.pkl'),'rb') as f:
        return pickle.load(f)

def input_fn(body, content_type='application/json'):
    if content_type != 'application/json':
        raise ValueError(f'Unsupported content type: {content_type}')
    payload = json.loads(body)
    if isinstance(payload, dict): payload = [payload]
    df = pd.DataFrame(payload)
    missing = set(FEATURE_COLUMNS) - set(df.columns)
    if missing: raise ValueError(f'Missing features: {missing}')
    return df[FEATURE_COLUMNS]

def predict_fn(data, model):
    return model.predict(data), model.predict_proba(data)[:,1]

def output_fn(prediction, accept='application/json'):
    preds, probas = prediction
    response = [{'prediction':int(p),
                 'label':'Heart disease present' if p==1 else 'No heart disease',
                 'probability':round(float(b),4)}
                for p,b in zip(preds,probas)]
    return json.dumps(response), accept


Overwriting src/inference.py


In [16]:
# No MLflow requirements file is needed in the SageMaker training container.
# MLflow logging is done after the pipeline completes, from this notebook.

print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")


Scripts written:
  src/preprocess.py  (3098 bytes)
  src/train.py  (4205 bytes)
  src/inference.py  (1256 bytes)


## 1A. Upload pipeline source files to S3

This section pre-places the three pipeline source files in the team S3 area.

Files uploaded:

```text
preprocess.py
train.py
inference.py
```

There is no `requirements_train.txt`: the SageMaker training container does not need MLflow or MLflow App packages. MLflow logging happens later from this notebook to the SageMaker MLflow App.

S3 location example:

```text
s3://nyp-26s1-iti113/iti113/team40/data/heart-disease/pipeline_src/
```


In [17]:
from pathlib import Path

s3_client = boto3.client("s3")

SOURCE_DIR = Path("src")
FILES_TO_UPLOAD = [
    "preprocess.py",
    "train.py",
    "inference.py",
]

for filename in FILES_TO_UPLOAD:
    local_path = SOURCE_DIR / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)

Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team07/data/heart-disease/pipeline_src/preprocess.py
Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team07/data/heart-disease/pipeline_src/train.py
Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team07/data/heart-disease/pipeline_src/inference.py
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team07/data/heart-disease/pipeline_src


## 1B. Download pipeline source files from S3

The SageMaker Pipeline below will use the local `pipeline_src/` folder, but that folder is recreated by downloading the source files from S3.

This verifies that the pipeline is using the S3-preplaced source files rather than directly depending on the original notebook-generated `src/` files.

In [18]:
import shutil

local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Downloaded s3://nyp-26s1-iti113/iti113/team07/data/heart-disease/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team07/data/heart-disease/pipeline_src/train.py -> pipeline_src/train.py
Downloaded s3://nyp-26s1-iti113/iti113/team07/data/heart-disease/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded files:
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/train.py


## 2. Define the SageMaker Pipeline

Four SageMaker Pipeline steps:

1. **ProcessingStep** — runs `preprocess.py`, outputs train/test CSVs to S3.
2. **TrainingStep** — runs `train.py`, prints metrics for SageMaker and saves the model artefact.
3. **ConditionStep** — checks AUC ≥ threshold before allowing registration.
4. **ModelStep** — registers the model in SageMaker Model Registry (`PendingManualApproval`).

MLflow is not inside the pipeline. A later notebook section logs the completed SageMaker run into the team SageMaker MLflow App experiment.


In [19]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

pipeline_session = PipelineSession()

# Pipeline parameters — can be overridden at execution time
p_n_est    = ParameterInteger(name='NEstimators',    default_value=200)
p_depth    = ParameterInteger(name='MaxDepth',       default_value=6)
p_samples  = ParameterInteger(name='MinSamplesLeaf', default_value=4)
p_gate     = ParameterFloat(  name='QualityGateAUC', default_value=QUALITY_GATE_AUC)

print('Pipeline parameters defined.')


Pipeline parameters defined.


In [20]:
# Step 1: ProcessingStep
processor = SKLearnProcessor(
    framework_version='1.2-1', instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1, role=role, sagemaker_session=pipeline_session,
    base_job_name=f'iti113-{TEAM_ID}-{STUDENT_ID}-process')

step_process = ProcessingStep(
    name='PreprocessData',
    processor=processor,
    inputs=[ProcessingInput(source=RAW_DATA_URI,
                            destination='/opt/ml/processing/input')],
    outputs=[ProcessingOutput(output_name='processed',
                              source='/opt/ml/processing/output',
                              destination=f'{PIPELINE_ROOT}/processed')],
    code=f'{LOCAL_PIPELINE_SRC}/preprocess.py',
    job_arguments=['--test-size','0.2','--random-state','42']
)
print('Step 1 (ProcessingStep) defined.')


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 1 (ProcessingStep) defined.


In [21]:
# Step 2: TrainingStep
# The training container receives no Databricks host, token, or MLflow dependency.
# It only trains the model and prints metrics for SageMaker to capture.
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "n-estimators": p_n_est,
        "max-depth": p_depth,
        "min-samples-leaf": p_samples,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "sagemaker_pipeline_run",
    },
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    metric_definitions=[
        {"Name": "test_auc_roc", "Regex": "Test AUC-ROC: ([0-9\\.]+)"},
        {"Name": "test_accuracy", "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_f1", "Regex": "test_f1: ([0-9\\.]+)"},
    ],
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        "train": sagemaker.inputs.TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),
        "test": sagemaker.inputs.TrainingInput(
            s3_data=processed_uri,
            content_type="text/csv"
        ),
    },
)

print("Step 2 (TrainingStep) defined.")


/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 2 (TrainingStep) defined.


In [22]:
# Step 3: ModelStep — register in SageMaker Model Registry
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_PIPELINE_SRC
)
step_register = ModelStep(
    name='RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('Step 3 (ModelStep) defined.')


/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Step 3 (ModelStep) defined.


In [23]:
# Step 4: ConditionStep — gate on SageMaker-captured test AUC
#
# The training script prints:
#     Test AUC-ROC: 0.xxxx
# and the estimator metric_definitions capture this as "test_auc_roc".
# This avoids relying on Databricks Model Registry or a separate evaluation file.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
    right=p_gate
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("Step 4 (ConditionStep) defined.")


Step 4 (ConditionStep) defined.


In [24]:
# Assemble and upsert the pipeline
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[p_n_est, p_depth, p_samples, p_gate],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session
)
pipeline.upsert(role_arn=role)
print(f'Pipeline "{PIPELINE_NAME}" upserted.')
print('View in SageMaker Studio: left sidebar -> Pipelines')


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team07-heart-disease" upserted.
View in SageMaker Studio: left sidebar -> Pipelines


## 3. Execute the Pipeline

In [25]:
execution = pipeline.start(parameters={
    'NEstimators':200, 'MaxDepth':6, 'MinSamplesLeaf':4, 'QualityGateAUC':0.75
})
print(f'Execution ARN: {execution.arn}')
print('Monitoring step status below. Takes ~10-15 minutes.')


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team07-heart-disease/execution/ikbg10jjnu4g
Monitoring step status below. Takes ~10-15 minutes.


In [26]:
import time

prev = {}

while True:

    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]

    steps = execution.list_steps()

    # Compatible with both old and new SageMaker SDKs
    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])

    for step in steps:
        n = step["StepName"]
        s = step["StepStatus"]

        if prev.get(n) != s:
            print(f"{n:<20} {s}")
            prev[n] = s

    if status in ("Succeeded", "Failed", "Stopped"):
        print(f"\nPipeline Status: {status}")
        break

    time.sleep(30)



PreprocessData       Starting


PreprocessData       Executing


TrainModel           Executing
PreprocessData       Succeeded


RegisterModel-RepackModel-0 Executing
AUCQualityGate       Succeeded
TrainModel           Succeeded


RegisterModel-RegisterModel Succeeded
RegisterModel-RepackModel-0 Succeeded

Pipeline Status: Succeeded


## 4. Log the Completed SageMaker Run to the SageMaker MLflow App

Run this section only after the pipeline has succeeded.

This notebook-side step reads the completed **SageMaker Training Job** to obtain:
- captured quality metrics,
- training hyperparameters,
- training-job name and pipeline execution ARN,
- SageMaker model-artifact S3 URI.

It then logs these as an MLflow run in the team experiment hosted by the **SageMaker Serverless MLflow App** created in Notebook 01. No Databricks host or token is required.


In [27]:
# Run only after the execution-monitoring cell reports "Pipeline Succeeded".
# In SageMaker SDK 2.257.3, execution.list_steps() returns a Python list.
# The compatibility helper also supports SDK versions that return a dictionary.
import mlflow


def get_pipeline_steps(execution):
    response = execution.list_steps()
    if isinstance(response, list):
        return response
    return response.get("PipelineExecutionSteps", [])


if execution.describe()["PipelineExecutionStatus"] != "Succeeded":
    raise RuntimeError(
        "The SageMaker Pipeline has not succeeded. "
        "Resolve pipeline failures before logging to MLflow."
    )

steps = get_pipeline_steps(execution)

print("Pipeline steps:")
for step in steps:
    print(f"  {step['StepName']}: {step['StepStatus']}")

train_step_info = next(
    (
        step for step in steps
        if step["StepName"] == "TrainModel"
        and step["StepStatus"] == "Succeeded"
    ),
    None
)

if train_step_info is None:
    raise RuntimeError(
        "A successful TrainModel step was not found in this pipeline execution."
    )

training_job_arn = train_step_info["Metadata"]["TrainingJob"]["Arn"]
training_job_name = training_job_arn.rsplit("/", 1)[-1]

sm_client = boto3.client("sagemaker", region_name=region)
training_job = sm_client.describe_training_job(
    TrainingJobName=training_job_name
)

# SageMaker captures the metrics printed by train.py through metric_definitions.
captured_metrics = {
    item["MetricName"]: float(item["Value"])
    for item in training_job.get("FinalMetricDataList", [])
    if item["MetricName"] in {"test_auc_roc", "test_accuracy", "test_f1"}
}

if not captured_metrics:
    raise RuntimeError(
        "No captured SageMaker metrics were found. "
        "Check train.py output and estimator.metric_definitions."
    )

model_artifact_s3_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]
training_hyperparameters = training_job.get("HyperParameters", {})

print("Training job:", training_job_name)
print("Model artefact:", model_artifact_s3_uri)
print("Captured metrics:", captured_metrics)

# Log to SageMaker Serverless MLflow App.
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow_run_name = (
    f"{TEAM_ID}_{STUDENT_ID}_sagemaker_pipeline_"
    f"{int(time.time())}"
)

with mlflow.start_run(run_name=mlflow_run_name) as run:
    mlflow.set_tags({
        "course": "ITI113",
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "dataset": "heart-disease",
        "execution_environment": "aws_sagemaker_pipeline",
        "tracking_backend": "sagemaker_mlflow_app",
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "sagemaker_model_artifact_s3_uri": model_artifact_s3_uri,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
    })

    # Hyperparameters arrive from SageMaker as strings, which are valid MLflow params.
    mlflow.log_params(training_hyperparameters)
    mlflow.log_metrics(captured_metrics)

    # Store a small, portable traceability record as an MLflow artefact.
    run_summary = {
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "training_job_arn": training_job_arn,
        "model_artifact_s3_uri": model_artifact_s3_uri,
        "metrics": captured_metrics,
        "hyperparameters": training_hyperparameters,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "semester": SEMESTER,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
        "tracking_backend": "sagemaker_mlflow_app",
    }

    summary_file = "sagemaker_pipeline_run_summary.json"
    with open(summary_file, "w") as f:
        json.dump(run_summary, f, indent=2)

    mlflow.log_artifact(
        summary_file,
        artifact_path="sagemaker_pipeline"
    )

    mlflow_run_id = run.info.run_id
    mlflow_experiment_id = run.info.experiment_id

print("SageMaker MLflow App logging completed.")
print("MLflow run ID:", mlflow_run_id)
print("MLflow App ARN:", MLFLOW_APP_ARN)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", mlflow_experiment_id)
print("\nIgnore any generic mlflow.sagemaker.app.aws link printed by MLflow above.")
print("Use the presigned links below instead:")
print_mlflow_presigned_links(
    experiment_id=mlflow_experiment_id,
    run_id=mlflow_run_id
)


Pipeline steps:
  RegisterModel-RegisterModel: Succeeded
  RegisterModel-RepackModel-0: Succeeded
  AUCQualityGate: Succeeded
  TrainModel: Succeeded
  PreprocessData: Succeeded


Training job: pipelines-ikbg10jjnu4g-TrainModel-naRSj8axSL
Model artefact: s3://sagemaker-ap-southeast-1-044528205969/pipelines-ikbg10jjnu4g-TrainModel-naRSj8axSL/output/model.tar.gz
Captured metrics: {'test_auc_roc': 0.9696999788284302, 'test_accuracy': 0.885200023651123, 'test_f1': 0.8813999891281128}


🏃 View run team07_s703_sagemaker_pipeline_1784877636 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/6fccdb0f7de4472a8b6545493289f6db
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
SageMaker MLflow App logging completed.
MLflow run ID: 6fccdb0f7de4472a8b6545493289f6db
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O
Experiment: ITI113/team07/Experiment1
Experiment ID: 1

Ignore any generic mlflow.sagemaker.app.aws link printed by MLflow above.
Use the presigned links below instead:


Presigned MLflow experiment URL:
https://app-PVSI6X27672O.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6Ik9IQlpaSyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNGdUT1BjOUwrOHVqUFdiTm9ybDVUYzFvbEk4M1A1T0VSZWNWeUs1ekxER3NBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFME5VaEliSFJxVDAxTlJXcENlWFJzV205RWVFSnNSbUYyT0Vka1RESnJOVFZXVmtSNUt6WnVNV1I1T0VoMk5HUTNSWFpMVTJFdlJHUTFSVU41WTFKV2R6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFZVTlJU1dieXlQdXVQaTV6eVBub2hvQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4enpLcUttN1FTWXBuRXEvOENBUkNBTzhlZ0k0aTUycndneFRLZndTbHpKZzdlR2FqQ0J4SUIzN2NxdVFpeERCWHhEL3hjbDFjVCtkcENQbE1yU3lUbmdtcmpVc1JWODhzdzRDMG9BZ0FBRUFET2dqOWJGRkd6RGExWHpqWFNHWDByOVlacVRkY3kvU3pocnZMbDhvSFlaMjhZZ1g0NGFFSk


Presigned MLflow run URL:
https://app-PVSI6X27672O.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkRBTUxZRyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNElRWFdSTDNzeitIcVd5N3hyT1F0VDJ4ODRqU0VodXF0QU0xZ0pxQXB2R2tBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGd2MwMWlWM1U0UWtZemNIcHNiMlZHWWpkcmNWY3pjRVJTZDJoSGVWVmxObGRSZGt0RVZrRjJZMWxrY1ZkWldUZ3dURGczUXpkNFlsUTRNSFpuUTNGelFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFZK21XdnQyVCt0bFRsZEV2T2JLcENjQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF5encvQnRsSXh0akJ6S2xEZ0NBUkNBTy9PcXFJRHJJUXpqTjM1T2d2Y3M5TmhnbUNpVEE0ZlFHUmVTVVczY1N5SzhMQ2Q5dTAza29abkpGeEcrTzBHTWVIVlhvYXRNWFJobDQ2aHlBZ0FBRUFDOHNRYnZkRU0xYytqSU0vUUdEZG9VLzF3Z1hWb2M3NXlFN2lwS1pZZzJmb0ZyYnVUYkNycU9QYVFp

## 5. Deploy Serverless Endpoint

After the pipeline succeeds, the model sits in Model Registry with `PendingManualApproval`.
We approve it here, then deploy as a **Serverless Endpoint** — cost is near-zero when idle.


In [29]:
sm = boto3.client('sagemaker')

# Get the latest registered model package
pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy='CreationTime', SortOrder='Descending', MaxResults=1
)['ModelPackageSummaryList']

if not pkgs:
    print('No model packages found. Check the pipeline completed the Register step.')
else:
    pkg_arn = pkgs[0]['ModelPackageArn']
    print(f'Model package : {pkg_arn}')
    print(f'Status        : {pkgs[0]["ModelApprovalStatus"]}')


Model package : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-HeartDisease/1
Status        : PendingManualApproval


In [30]:
# Approve the model
sm.update_model_package(ModelPackageArn=pkg_arn, ModelApprovalStatus='Approved')
print(f'Approved: {pkg_arn}')


Approved: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team07-HeartDisease/1


In [37]:
from sagemaker import ModelPackage
from sagemaker.serverless import ServerlessInferenceConfig

deployable = ModelPackage(
    role=role, model_package_arn=pkg_arn, sagemaker_session=sagemaker.Session())

serverless_cfg = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=5)

print(f'Deploying serverless endpoint: {ENDPOINT_NAME}')
print('This takes 3-5 minutes...')
predictor = deployable.deploy(
    serverless_inference_config=serverless_cfg,
    endpoint_name=ENDPOINT_NAME
)
print(f'Endpoint ready: {ENDPOINT_NAME}')
print('Cost: ~$0 idle. Charged per invocation only.')


INFO:sagemaker:Creating model with name: team07-HeartDisease-2026-07-24-07-30-06-026


Deploying serverless endpoint: iti113-team07-heart-disease
This takes 3-5 minutes...


INFO:sagemaker:Creating endpoint-config with name iti113-team07-heart-disease


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8                                                                                             │
│    9 print(f'Deploying serverless endpoint: {ENDPOINT_NAME}')                                    │
│   10 print('This takes 3-5 minutes...')                                                          │
│ ❱ 11 predictor = deployable.deploy(                                                              │
│   12 │   serverless_inference_config=serverless_cfg,                                             │
│   13 │   endpoint_name=ENDPOINT_NAME                                                             │
│   14 )                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model.py:1837 in deploy                        │
│                                                                                                  │
│   1834 │   │   │   │   )                                                                         │
│   1835 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1836 │   │   │   else:                                                                         │
│ ❱ 1837 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1838 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1839 │   │   │   │   │   production_variants=[production_variant],                             │
│   1840 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:6357 in                             │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6354 │   │   │   config_options["ExecutionRoleArn"] = role                                     │
│   6355 │   │                                                                                     │
│   6356 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│ ❱ 6357 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6358 │   │                                                                                     │
│   6359 │   │   return self.create_endpoint(                                                      │
│   6360 │   │   │   endpoint_name=name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name) 

## 6. Test the Live Endpoint

In [32]:
rt = boto3.client('sagemaker-runtime')

# High-risk profile: 55yo male with several cardiac risk factors
high_risk = {
    'age':55,'trestbps':140,'chol':250,'thalach':145,'oldpeak':2.3,
    'sex':1,'cp':0,'fbs':0,'restecg':0,'exang':1,
    'slope':1,'ca':1,'thal':7,'age_group':1,'high_risk_count':2
}

resp = rt.invoke_endpoint(EndpointName=ENDPOINT_NAME,
                           ContentType='application/json',
                           Body=json.dumps(high_risk))
result = json.loads(resp['Body'].read())[0]
print('HIGH-RISK PROFILE (55yo male, asymptomatic CP, exercise angina)')
print(f'  Prediction  : {result["label"]}')
print(f'  Probability : {result["probability"]:.1%}')


HIGH-RISK PROFILE (55yo male, asymptomatic CP, exercise angina)
  Prediction  : Heart disease present
  Probability : 65.5%


In [33]:
# Low-risk profile for comparison
low_risk = {
    'age':35,'trestbps':120,'chol':190,'thalach':170,'oldpeak':0.5,
    'sex':0,'cp':2,'fbs':0,'restecg':0,'exang':0,
    'slope':2,'ca':0,'thal':3,'age_group':0,'high_risk_count':0
}
resp2 = rt.invoke_endpoint(EndpointName=ENDPOINT_NAME,
                            ContentType='application/json',
                            Body=json.dumps(low_risk))
result2 = json.loads(resp2['Body'].read())[0]
print('LOW-RISK PROFILE (35yo female, non-anginal CP, normal thal, no risk factors)')
print(f'  Prediction  : {result2["label"]}')
print(f'  Probability : {result2["probability"]:.1%}')


LOW-RISK PROFILE (35yo female, non-anginal CP, normal thal, no risk factors)
  Prediction  : No heart disease
  Probability : 27.2%


#### Delete endpoint when no lonnger needed (below code)

In [34]:
# To delete endpoint when no longer needed, uncomment:
boto3.client('sagemaker').delete_endpoint(EndpointName=ENDPOINT_NAME)
print(f'Endpoint deleted: {ENDPOINT_NAME}')


Endpoint deleted: iti113-team07-heart-disease


In [35]:
print('=' * 55)
print('NOTEBOOK 03 COMPLETE')
print('=' * 55)
print(f'Pipeline : {PIPELINE_NAME}')
print(f'MLflow   : {MLFLOW_EXPERIMENT_NAME} on SageMaker MLflow App')
print(f'MLflow App ARN : {MLFLOW_APP_ARN}')
print(f'SageMaker Registry : {MODEL_PACKAGE_GROUP}')
print(f'Endpoint : {ENDPOINT_NAME} (Serverless)')
print()
print('Next: Notebook 04 — AI Governance, Bias & Explainability')


NOTEBOOK 03 COMPLETE
Pipeline : iti113-team07-heart-disease
MLflow   : ITI113/team07/Experiment1 on SageMaker MLflow App
MLflow App ARN : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O
SageMaker Registry : team07-HeartDisease
Endpoint : iti113-team07-heart-disease (Serverless)

Next: Notebook 04 — AI Governance, Bias & Explainability


## 6. GitHub Actions CI/CD [SKIP THIS]

Save the workflow below as `.github/workflows/run_pipeline.yml` in your repository.  
It will re-run the pipeline automatically on every push to `main`.

**Setup:** GitHub → Settings → Secrets → Actions → add `AWS_ROLE_ARN` and `SAGEMAKER_BUCKET`. MLflow App permissions are not needed by the SageMaker pipeline itself unless the CI job also performs post-pipeline MLflow logging.


In [ ]:
workflow_yaml = '''
name: Run SageMaker MLOps Pipeline

on:
  push:
    branches: [ main ]
    paths: [ 'src/**', 'pipeline/**' ]

permissions:
  id-token: write
  contents: read

jobs:
  run-pipeline:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.10' }
      - run: pip install sagemaker boto3
      - uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: ${{ secrets.AWS_ROLE_ARN }}
          aws-region: ap-southeast-1
      - env:
          SAGEMAKER_BUCKET: ${{ secrets.SAGEMAKER_BUCKET }}
        run: python pipeline/run_pipeline.py --pipeline-name iti113-T01-heart-disease --wait
'''
print(workflow_yaml)
# Optionally save to disk:
# os.makedirs('.github/workflows', exist_ok=True)
# open('.github/workflows/run_pipeline.yml','w').write(workflow_yaml.strip())


---
## Checklist before Notebook 04

- [ ] `src/preprocess.py`, `train.py`, `inference.py` written and reviewed
- [ ] Pipeline upserted (visible in SageMaker Studio under Pipelines)
- [ ] Pipeline execution completed (all steps green)
- [ ] Post-pipeline MLflow run visible in the team SageMaker MLflow App experiment
- [ ] Model registered in SageMaker Model Registry after passing AUC gate
- [ ] Serverless Endpoint deployed and responding to test calls
- [ ] Both high-risk and low-risk test profiles return sensible predictions


### Below Code Gets the model from endpoint (make sure endpoint is not deleted)

In [26]:
import os
import json
import boto3
from urllib.parse import urlparse
from pathlib import Path
from botocore.exceptions import ClientError


# ============================================================
# CONFIGURATION
# ============================================================

REGION = "ap-southeast-1"

# Your deployed serverless endpoint name
ENDPOINT_NAME = ENDPOINT_NAME #"REPLACE_WITH_YOUR_ENDPOINT_NAME"

# Your class/team bucket and desired destination folder
DESTINATION_BUCKET = "nyp-26s1-iti113" # "REPLACE_WITH_YOUR_CLASS_BUCKET"

# Recommended destination naming
DESTINATION_KEY = (
    f"iti113/{TEAM_ID}/models/"
    f"{PROJECT_NAME}/"
    "model-package-v2/"
    "model.tar.gz"
)

# Optional local notebook download folder
LOCAL_DOWNLOAD_DIR = Path("downloaded_models")


# ============================================================
# AWS CLIENTS
# ============================================================

sm_client = boto3.client("sagemaker", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)
s3_resource = boto3.resource("s3", region_name=REGION)


# ============================================================
# HELPER: Parse an S3 URI
# ============================================================

def parse_s3_uri(s3_uri: str):
    """
    Convert:
        s3://bucket-name/path/to/object
    into:
        bucket-name, path/to/object
    """
    parsed = urlparse(s3_uri)

    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")

    return parsed.netloc, parsed.path.lstrip("/")


# ============================================================
# STEP 1: Endpoint -> Endpoint Config -> SageMaker Model
# ============================================================

endpoint_desc = sm_client.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)

endpoint_config_name = endpoint_desc["EndpointConfigName"]

endpoint_config_desc = sm_client.describe_endpoint_config(
    EndpointConfigName=endpoint_config_name
)

production_variants = endpoint_config_desc["ProductionVariants"]

if not production_variants:
    raise ValueError("No production variants found in endpoint configuration.")

model_name = production_variants[0]["ModelName"]

model_desc = sm_client.describe_model(
    ModelName=model_name
)

print("Endpoint name:", ENDPOINT_NAME)
print("Endpoint status:", endpoint_desc["EndpointStatus"])
print("Endpoint configuration:", endpoint_config_name)
print("SageMaker model:", model_name)


# ============================================================
# STEP 2: SageMaker Model -> Model Package
# ============================================================

containers = model_desc.get("Containers", [])

if not containers:
    raise ValueError(
        "No Containers found in SageMaker model definition. "
        "Expected a model created from a Model Package."
    )

model_package_arn = containers[0].get("ModelPackageName")

if not model_package_arn:
    raise ValueError(
        "This model does not contain ModelPackageName. "
        "Inspect model_desc manually for a direct ModelDataUrl."
    )

print("Model package ARN:", model_package_arn)

package_desc = sm_client.describe_model_package(
    ModelPackageName=model_package_arn
)

package_containers = package_desc["InferenceSpecification"]["Containers"]

if not package_containers:
    raise ValueError("No inference containers found in model package.")

model_s3_uri = package_containers[0].get("ModelDataUrl")

if not model_s3_uri:
    raise ValueError(
        "ModelDataUrl not found in model package inference container.\n"
        + json.dumps(package_containers[0], indent=2, default=str)
    )

print("\nOriginal model artefact S3 URI:")
print(model_s3_uri)


# ============================================================
# STEP 3: Verify source object exists
# ============================================================

source_bucket, source_key = parse_s3_uri(model_s3_uri)

source_metadata = s3_client.head_object(
    Bucket=source_bucket,
    Key=source_key
)

source_size_mb = source_metadata["ContentLength"] / (1024 * 1024)

print("\nSource bucket:", source_bucket)
print("Source key:", source_key)
print(f"Source model size: {source_size_mb:.2f} MB")


# ============================================================
# STEP 4: Download locally to the Studio notebook environment
# ============================================================

LOCAL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

local_model_file = LOCAL_DOWNLOAD_DIR / "model.tar.gz"

print("\nDownloading model locally...")
s3_client.download_file(
    source_bucket,
    source_key,
    str(local_model_file)
)

print("Local file saved to:")
print(local_model_file.resolve())

print(f"Local file size: {local_model_file.stat().st_size / (1024 * 1024):.2f} MB")


# ============================================================
# STEP 5: Copy the model directly into your team S3 prefix
# ============================================================

copy_source = {
    "Bucket": source_bucket,
    "Key": source_key
}

print("\nCopying model into team S3 folder...")

s3_resource.meta.client.copy(
    CopySource=copy_source,
    Bucket=DESTINATION_BUCKET,
    Key=DESTINATION_KEY
)

destination_s3_uri = f"s3://{DESTINATION_BUCKET}/{DESTINATION_KEY}"

print("\nCopy completed successfully.")
print("Destination model artefact:")
print(destination_s3_uri)


# ============================================================
# STEP 6: Verify destination object
# ============================================================

destination_metadata = s3_client.head_object(
    Bucket=DESTINATION_BUCKET,
    Key=DESTINATION_KEY
)

print("\nDestination verification:")
print("Destination size (MB):", round(
    destination_metadata["ContentLength"] / (1024 * 1024),
    2
))
print("Last modified:", destination_metadata["LastModified"])
print("ETag:", destination_metadata["ETag"])


Endpoint name: iti113-team40-heart-disease
Endpoint status: InService
Endpoint configuration: iti113-team40-heart-disease
SageMaker model: team40-HeartDisease-2026-07-11-15-51-36-263
Model package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team40-HeartDisease/1

Original model artefact S3 URI:
s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-07-11-15-35-47-349/pipelines-vonkbsbsyriz-RegisterModel-Repack-d1n8PIvKc9/output/model.tar.gz

Source bucket: sagemaker-ap-southeast-1-044528205969
Source key: sagemaker-scikit-learn-2026-07-11-15-35-47-349/pipelines-vonkbsbsyriz-RegisterModel-Repack-d1n8PIvKc9/output/model.tar.gz
Source model size: 0.17 MB

Local file saved to:
/home/sagemaker-user/downloaded_models/model.tar.gz
Local file size: 0.17 MB

Copying model into team S3 folder...



Copy completed successfully.
Destination model artefact:
s3://nyp-26s1-iti113/iti113/team40/models/heart-disease/model-package-v2/model.tar.gz

Destination verification:
Destination size (MB): 0.17
Last modified: 2026-07-11 15:55:38+00:00
ETag: "4a0e9ba128ed2e17659e24fc4d2918d7"
